In [ ]:
import warnings
warnings.filterwarnings("ignore")

# ViT와 위치 임베딩 보간 함수 불러오기
from models.vit import VisionTransformer, interpolate_pos_embed
# BERT 기반 구성 및 모델 (텍스트 인코더/디코더) 불러오기
from models.med import BertConfig, BertModel, BertLMHeadModel
# 사전학습된 BERT 토크나이저를 가져옴
from transformers import BertTokenizer

import torch
from torch import nn
import torch.nn.functional as F

import os
from urllib.parse import urlparse
from timm.models.hub import download_cached_file

# BLIP Base: 이미지, 텍스트 또는 둘을 함께 사용하는 멀티모달 특성 추출기
class BLIP_Base(nn.Module):
    def __init__(self,
                 med_config='configs/med_config.json',  # BERT 설정 JSON 파일 경로
                 image_size=224,                        # Vision Transformer 입력 이미지 크기
                 vit='base',                            # 사용할 ViT 모델 크기 ('base' or 'large' 중에 선택 가능)
                 vit_grad_ckpt=False,                   # 메모리 절약을 위한 gradient checkpointing 여부 설정
                 vit_ckpt_layer=0):                     # checkpoint를 적용할 레이어 수
        super().__init__()

        # Vision Transformer 생성 및 출력 임베딩 차원 확인
        self.visual_encoder, vision_width = create_vit(vit, image_size, vit_grad_ckpt, vit_ckpt_layer)

        # BLIP 모델은 텍스트와 이미지를 연결하는 모델이므로 BERT 토크나이저가 필요
        self.tokenizer = init_tokenizer()

        # BERT 설정 로드 및 vision encoder 출력 크기와 일치하도록 설정 수정
        med_config = BertConfig.from_json_file(med_config)
        med_config.encoder_width = vision_width

        # 텍스트 인코더는 pooling 없이 hidden states 전체를 반환
        self.text_encoder = BertModel(config=med_config, add_pooling_layer=False)

    def forward(self, image, caption, mode):
        # 사용할 모드는 'image', 'text', 'multimodal' 중 하나
        assert mode in ['image', 'text', 'multimodal'], "mode parameter must be image, text, or multimodal"

        # caption을 토큰화하여 텐서로 변환하고 이미지와 같은 디바이스로 이동
        text = self.tokenizer(caption, return_tensors="pt").to(image.device)

        if mode == 'image':
            # 이미지 임베딩만 반환
            image_embeds = self.visual_encoder(image)
            return image_embeds

        elif mode == 'text':
            # 텍스트 임베딩만 반환
            text_output = self.text_encoder(text.input_ids, attention_mask=text.attention_mask,
                                            return_dict=True, mode='text')
            return text_output.last_hidden_state

        elif mode == 'multimodal':
            # 이미지 임베딩을 얻고 어텐션 마스크 생성
            image_embeds = self.visual_encoder(image)
            image_atts = torch.ones(image_embeds.size()[:-1], dtype=torch.long).to(image.device)

            # 텍스트의 첫 토큰을 [ENC] 토큰으로 변경하여 인코딩 입력으로 설정
            text.input_ids[:, 0] = self.tokenizer.enc_token_id

            # 텍스트 인코더에 이미지 임베딩을 encoder_hidden_states로 제공하여 멀티모달 처리
            output = self.text_encoder(text.input_ids,
                                       attention_mask=text.attention_mask,
                                       encoder_hidden_states=image_embeds,
                                       encoder_attention_mask=image_atts,
                                       return_dict=True)
            return output.last_hidden_state

# BLIP Decoder: 이미지에서 캡션을 생성하는 데 사용되는 모델
class BLIP_Decoder(nn.Module):
    def __init__(self,
                 med_config='configs/med_config.json',
                 image_size=384,
                 vit='base',
                 vit_grad_ckpt=False,
                 vit_ckpt_layer=0,
                 prompt='a picture of '):  # 캡션 생성을 위한 기본 프롬프트
        super().__init__()

        # 비전 인코더 초기화
        self.visual_encoder, vision_width = create_vit(vit, image_size, vit_grad_ckpt, vit_ckpt_layer)

        # 텍스트 토크나이저 초기화
        self.tokenizer = init_tokenizer()

        # 텍스트 디코더 설정 구성 및 초기화
        med_config = BertConfig.from_json_file(med_config)
        med_config.encoder_width = vision_width
        self.text_decoder = BertLMHeadModel(config=med_config)

        self.prompt = prompt
        self.prompt_length = len(self.tokenizer(self.prompt).input_ids) - 1  # 프롬프트 길이 계산

    def forward(self, image, caption):
        # 이미지 인코딩 및 어텐션 마스크 생성
        image_embeds = self.visual_encoder(image)
        image_atts = torch.ones(image_embeds.size()[:-1], dtype=torch.long).to(image.device)

        # 캡션 토큰화 및 첫 토큰을 [DEC] 토큰으로 설정
        text = self.tokenizer(caption, padding='longest', truncation=True, max_length=40, return_tensors="pt").to(image.device)
        text.input_ids[:, 0] = self.tokenizer.bos_token_id

        # 손실 계산을 위한 디코더 타겟 생성
        decoder_targets = text.input_ids.masked_fill(text.input_ids == self.tokenizer.pad_token_id, -100)
        decoder_targets[:, :self.prompt_length] = -100

        # 텍스트 디코더 forward 수행
        decoder_output = self.text_decoder(text.input_ids,
                                           attention_mask=text.attention_mask,
                                           encoder_hidden_states=image_embeds,
                                           encoder_attention_mask=image_atts,
                                           labels=decoder_targets,
                                           return_dict=True)
        return decoder_output.loss

    def generate(self, image, sample=False, num_beams=3, max_length=30, min_length=10, top_p=0.9, repetition_penalty=1.0):
        # 이미지 임베딩 추출
        image_embeds = self.visual_encoder(image)

        # Beam search일 경우 이미지 임베딩을 beam 수만큼 복제
        if not sample:
            image_embeds = image_embeds.repeat_interleave(num_beams, dim=0)

        image_atts = torch.ones(image_embeds.size()[:-1], dtype=torch.long).to(image.device)
        model_kwargs = {"encoder_hidden_states": image_embeds, "encoder_attention_mask": image_atts}

        # 프롬프트 입력 설정
        prompt = [self.prompt] * image.size(0)
        input_ids = self.tokenizer(prompt, return_tensors="pt").input_ids.to(image.device)
        input_ids[:, 0] = self.tokenizer.bos_token_id
        input_ids = input_ids[:, :-1]  # 마지막 토큰 제외

        if sample:
            # Nucleus Sampling 방식으로 문장 생성(일정 기준 이상의 누적확률 중 랜덤으로 뽑는 것)
            outputs = self.text_decoder.generate(input_ids=input_ids,
                                                 max_length=max_length,
                                                 min_length=min_length,
                                                 do_sample=True,
                                                 top_p=top_p,
                                                 num_return_sequences=1,
                                                 eos_token_id=self.tokenizer.sep_token_id,
                                                 pad_token_id=self.tokenizer.pad_token_id,
                                                 repetition_penalty=1.1,
                                                 **model_kwargs)
        else:
            # Beam Search 방식으로 문장 생성(누적 확률이 최대인 것을 뽑는 것)
            outputs = self.text_decoder.generate(input_ids=input_ids,
                                                 max_length=max_length,
                                                 min_length=min_length,
                                                 num_beams=num_beams,
                                                 eos_token_id=self.tokenizer.sep_token_id,
                                                 pad_token_id=self.tokenizer.pad_token_id,
                                                 repetition_penalty=repetition_penalty,
                                                 **model_kwargs)

        # 생성된 토큰을 디코딩하고 프롬프트 부분 제거
        captions = []
        for output in outputs:
            caption = self.tokenizer.decode(output, skip_special_tokens=True)
            captions.append(caption[len(self.prompt):])
        return captions

# BLIP 디코더 인스턴스 생성 함수
def blip_decoder(pretrained='', **kwargs):
    model = BLIP_Decoder(**kwargs)
    if pretrained:
        model, msg = load_checkpoint(model, pretrained)
        assert len(msg.missing_keys) == 0
    return model

# BLIP 피처 추출기 인스턴스 생성 함수
def blip_feature_extractor(pretrained='', **kwargs):
    model = BLIP_Base(**kwargs)
    if pretrained:
        model, msg = load_checkpoint(model, pretrained)
        assert len(msg.missing_keys) == 0
    return model

# BERT 토크나이저 초기화 함수
def init_tokenizer():
    tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
    tokenizer.add_special_tokens({'bos_token': '[DEC]'})
    tokenizer.add_special_tokens({'additional_special_tokens': ['[ENC]']})
    tokenizer.enc_token_id = tokenizer.additional_special_tokens_ids[0]
    return tokenizer

# Vision Transformer 모델 생성 함수
def create_vit(vit, image_size, use_grad_checkpointing=False, ckpt_layer=0, drop_path_rate=0):
    assert vit in ['base', 'large'], "vit parameter must be base or large"

    if vit == 'base':
        vision_width = 768
        visual_encoder = VisionTransformer(img_size=image_size, patch_size=16, embed_dim=vision_width, depth=12,
                                           num_heads=12, use_grad_checkpointing=use_grad_checkpointing, ckpt_layer=ckpt_layer,
                                           drop_path_rate=drop_path_rate)
    elif vit == 'large':
        vision_width = 1024
        visual_encoder = VisionTransformer(img_size=image_size, patch_size=16, embed_dim=vision_width, depth=24,
                                           num_heads=16, use_grad_checkpointing=use_grad_checkpointing, ckpt_layer=ckpt_layer,
                                           drop_path_rate=drop_path_rate or 0.1)
    return visual_encoder, vision_width

# 문자열이 URL인지 파일 경로인지 판단하는 함수
def is_url(url_or_filename):
    parsed = urlparse(url_or_filename)
    return parsed.scheme in ("http", "https")

# 모델 체크포인트 로드 함수
def load_checkpoint(model, url_or_filename):
    if is_url(url_or_filename):
        cached_file = download_cached_file(url_or_filename, check_hash=False, progress=True)
        checkpoint = torch.load(cached_file, map_location='cpu')
    elif os.path.isfile(url_or_filename):
        checkpoint = torch.load(url_or_filename, map_location='cpu')
    else:
        raise RuntimeError('checkpoint url or path is invalid')

    state_dict = checkpoint['model']

    # 포지션 임베딩 크기 차이에 따른 보간
    state_dict['visual_encoder.pos_embed'] = interpolate_pos_embed(state_dict['visual_encoder.pos_embed'], model.visual_encoder)
    if 'visual_encoder_m.pos_embed' in model.state_dict().keys():
        state_dict['visual_encoder_m.pos_embed'] = interpolate_pos_embed(state_dict['visual_encoder_m.pos_embed'], model.visual_encoder_m)

    # 모델 구조와 맞지 않는 키는 제거
    for key in model.state_dict().keys():
        if key in state_dict.keys() and state_dict[key].shape != model.state_dict()[key].shape:
            del state_dict[key]

    msg = model.load_state_dict(state_dict, strict=False)
    print('load checkpoint from %s' % url_or_filename)
    return model, msg